# YouTube Analytics — Data Cleaning & Feature Engineering

**Tujuan Notebook:**
1. Pembersihan data mentah dari YouTube Studio
2. Engineering fitur-fitur penting untuk ML
3. Klasifikasi performa video (performance class)

---

## 0. Setup & Import

In [52]:
import pandas as pd
import numpy as np
import re
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("✅ Library siap!")

✅ Library siap!


## 1. Load Data

In [53]:
# ─── Ganti path sesuai lokasi file kamu ───
FILE_PATH = 'data_fix.csv'

df_raw = pd.read_csv(FILE_PATH)
print(f"Raw shape: {df_raw.shape}")
df_raw.head(3)

Raw shape: (3012, 73)


,Konten,Judul video,Waktu publikasi video,Durasi,Penayangan tak dilewati,Rata-rata durasi tonton,Persentase penayangan rata-rata (%),Tetap menonton (%),Penonton unik,Penayangan rata-rata per penonton,Penonton baru,Penonton yang kembali,Penonton biasa,Penonton reguler,Diviralkan,Poin hype,Subscriber yang diperoleh,Subscriber yang hilang,Suka,Tidak suka,Suka (vs. tidak suka) (%),Pembagian,Komentar ditambahkan,Pendapatan transaksi (IDR),Transaksi,...,Tayangan per playlist dimulai,Jam streaming,Set pengingat,Pesan chat,Reaksi,Perubahan jumlah subscriber per postingan,Jumlah remix,Penayangan remix,Penayangan klip komunitas,Waktu tonton dari klip komunitas (jam),Jumlah klik pada kartu,Klik per kartu yang ditampilkan (%),Kartu ditampilkan,Jumlah klik pada teaser kartu,Teaser kartu ditampilkan,Klik teaser per teaser kartu yang ditampilkan (%),Klik pada elemen layar akhir,Klik per elemen layar akhir yang ditampilkan (%),Elemen layar akhir yang ditampilkan,Penayangan,Waktu tonton (jam),Subscriber,Estimasi pendapatan (IDR),Tayangan,Rasio klik-tayang dari tayangan (%)
0,Total,NaN,NaN,NaN,"19,920,894.00",0:04:01,37.47,80.43,0.00,0.00,0.00,0.00,0.00,0.00,280.00,"23,980.00","39,422.00","8,485.00","260,795.00","6,042.00",97.74,"24,871.00","95,538.00","34,108.88",161.00,...,1.70,125.96,"4,793.00",192.00,750.00,49.00,17.00,"10,950.00",4.00,0.05,6.00,1.22,493.00,40.00,"11,354.00",0.35,"36,786.00",2.09,"1,757,819.00","20,166,890.00","1,336,489.13","30,937.00","73,682,866.49","126,067,471.00",13.08
1,KAqnuJ0W8Jo,NEGARA ARAB & MALAYSIA TERDIAM MELIHAT INDONES...,"Aug 23, 2025",740.00,"915,959.00",0:04:55,39.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,"5,516.00",138.00,"13,544.00",184.00,98.66,"2,284.00","1,840.00",0.00,0.00,...,NaN,NaN,0.00,NaN,NaN,0.00,3.00,"1,961.00",0.00,0.00,0.00,NaN,0.00,0.00,0.00,NaN,74.00,4.56,"1,622.00","915,959.00","75,207.69","5,378.00","4,195,851.50","5,722,428.00",11.54
2,a5VBaiz4yS0,INDONESIA BIKIN ISRAEL NGAMUK‼ SAAT ARAB TAKUT...,"Aug 24, 2025",649.00,"319,187.00",0:03:59,36.89,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.00,820.00,30.00,"3,033.00",43.00,98.60,287.00,435.00,0.00,0.00,...,NaN,NaN,0.00,NaN,NaN,0.00,NaN,13.00,0.00,0.00,0.00,NaN,0.00,0.00,0.00,NaN,0.00,NaN,0.00,"319,187.00","21,226.10",790.00,"1,052,102.11","1,487,638.00",14.30


---
## 2. Pembersihan Data (Data Cleaning)

### 2.1 Hapus baris Total & baris kosong

In [54]:
# Baris pertama adalah baris 'Total' dari YouTube Studio — hapus
df = df_raw[
    df_raw['Konten'].notna() & 
    (df_raw['Konten'] != 'Total')
].copy()

df = df.reset_index(drop=True)
print(f" Setelah hapus baris Total: {len(df)} video")

 Setelah hapus baris Total: 3006 video


### 2.2 Hapus kolom yang hampir semua null (>80%)

In [55]:
THRESHOLD_NULL = 0.80  # Kolom dengan >80% null akan di-drop

null_pct = df.isnull().mean()
cols_to_drop = null_pct[null_pct > THRESHOLD_NULL].index.tolist()

print(f"Kolom yang di-drop ({len(cols_to_drop)} kolom):")
for col in cols_to_drop:
    print(f"  - {col}: {null_pct[col]*100:.1f}% null")

df = df.drop(columns=cols_to_drop)
print(f"\n Shape setelah drop kolom null berat: {df.shape}")

Kolom yang di-drop (15 kolom):
  - Tetap menonton (%): 92.8% null
  - Penonton unik: 100.0% null
  - Penayangan rata-rata per penonton: 100.0% null
  - Penonton baru: 100.0% null
  - Penonton yang kembali: 100.0% null
  - Penonton biasa: 100.0% null
  - Penonton reguler: 100.0% null
  - Pendapatan per transaksi (IDR): 83.6% null
  - Waktu tonton playlist (jam): 81.2% null
  - Penayangan dari playlist: 81.2% null
  - Tayangan per playlist dimulai: 86.7% null
  - Jumlah remix: 97.2% null
  - Penayangan remix: 96.3% null
  - Klik per kartu yang ditampilkan (%): 100.0% null
  - Klik teaser per teaser kartu yang ditampilkan (%): 100.0% null

 Shape setelah drop kolom null berat: (3006, 58)


### 2.3 Konversi tipe data

In [56]:
# ── Konversi waktu tonton format MM:SS atau H:MM:SS → detik ──
def parse_duration_to_seconds(val):
    """Konversi string durasi '0:04:55' atau '4:55' ke detik (int)."""
    if pd.isna(val) or val == '':
        return np.nan
    val = str(val).strip()
    parts = val.split(':')
    try:
        parts = [int(p) for p in parts]
        if len(parts) == 3:   # H:MM:SS
            return parts[0]*3600 + parts[1]*60 + parts[2]
        elif len(parts) == 2: # MM:SS
            return parts[0]*60 + parts[1]
        else:
            return int(parts[0])
    except:
        return np.nan

# Rata-rata durasi tonton → detik
if 'Rata-rata durasi tonton' in df.columns:
    df['avg_watch_seconds'] = df['Rata-rata durasi tonton'].apply(parse_duration_to_seconds)
    print("avg_watch_seconds dibuat")

# Durasi video (sudah dalam detik di dataset ini)
df['durasi_detik'] = pd.to_numeric(df['Durasi'], errors='coerce')

# Waktu publikasi → datetime
if 'Waktu publikasi video' in df.columns:
    df['published_at'] = pd.to_datetime(df['Waktu publikasi video'], errors='coerce')
    print("published_at dibuat")

# Kolom numerik — pastikan float
numeric_cols = [
    'Penayangan', 'Tayangan', 'Suka', 'Tidak suka', 'Komentar ditambahkan',
    'Pembagian', 'Subscriber yang diperoleh', 'Subscriber yang hilang',
    'Poin hype', 'Estimasi pendapatan (IDR)', 'RPM (IDR)', 'CPM (IDR)',
    'Rasio klik-tayang dari tayangan (%)', 'Persentase penayangan rata-rata (%)',
    'Penonton unik', 'Waktu tonton (jam)'
]
existing_numeric = [c for c in numeric_cols if c in df.columns]
df[existing_numeric] = df[existing_numeric].apply(pd.to_numeric, errors='coerce')
print(f" {len(existing_numeric)} kolom numerik dikonversi")

avg_watch_seconds dibuat
published_at dibuat
 15 kolom numerik dikonversi


### 2.4 Hapus baris yang tidak memiliki data inti

In [57]:
# Kolom WAJIB ada nilainya
required_cols = ['Penayangan', 'durasi_detik', 'Tayangan']
existing_required = [c for c in required_cols if c in df.columns]

before = len(df)
df = df.dropna(subset=existing_required)
after = len(df)

print(f"  Hapus {before - after} baris tanpa data inti")
print(f"   Sisa: {after} video")

  Hapus 6 baris tanpa data inti
   Sisa: 3000 video


### 2.5 Hapus data outlier ekstrem (Penayangan = 0)

In [58]:
before = len(df)
df = df[df['Penayangan'] > 0]
print(f"  Hapus {before - len(df)} video dengan 0 penayangan")
print(f"   Sisa: {len(df)} video")

  Hapus 0 video dengan 0 penayangan
   Sisa: 3000 video


### 2.6 Ringkasan data bersih

In [59]:
print("=" * 50)
print(f" Data Bersih: {len(df)} video, {df.shape[1]} kolom")
print("=" * 50)

# Null summary setelah cleaning
null_remaining = df.isnull().mean() * 100
null_remaining = null_remaining[null_remaining > 0].sort_values(ascending=False)
print("\nNull yang masih tersisa (kolom yang ada null-nya):")
print(null_remaining.head(15))

 Data Bersih: 3000 video, 61 kolom

Null yang masih tersisa (kolom yang ada null-nya):
Klik per elemen layar akhir yang ditampilkan (%)   42.20
Jam streaming                                      21.07
Pesan chat                                         21.07
Reaksi                                             21.07
Pendapatan iklan YouTube (IDR)                      5.87
Tayangan iklan                                      5.87
CPM berbasis-pemutaran (IDR)                        5.87
CPM (IDR)                                           5.87
Estimasi pemutaran yang dimonetisasi                5.87
RPM (IDR)                                           0.67
dtype: float64


---
## 3. Feature Engineering

### 3.1 Engagement Ratios

In [60]:
views = df['Penayangan'].replace(0, np.nan)  # hindari division by zero

# Like rate, comment rate, share rate
if 'Suka' in df.columns:
    df['like_rate'] = df['Suka'] / views

if 'Komentar ditambahkan' in df.columns:
    df['comment_rate'] = df['Komentar ditambahkan'] / views

if 'Pembagian' in df.columns:
    df['share_rate'] = df['Pembagian'] / views

if 'Tidak suka' in df.columns and 'Suka' in df.columns:
    total_reactions = df['Suka'] + df['Tidak suka']
    df['dislike_ratio'] = df['Tidak suka'] / total_reactions.replace(0, np.nan)

# Overall engagement rate (like + comment + share per view)
engagement_cols = [c for c in ['Suka', 'Komentar ditambahkan', 'Pembagian'] if c in df.columns]
df['engagement_rate'] = df[engagement_cols].sum(axis=1) / views

print("   Engagement ratios dibuat:")
print("   like_rate, comment_rate, share_rate, dislike_ratio, engagement_rate")

   Engagement ratios dibuat:
   like_rate, comment_rate, share_rate, dislike_ratio, engagement_rate


### 3.2 Watch Time & Retention Metrics

In [61]:
# Watch Time Ratio: rata-rata durasi tonton / durasi video
# → 1.0 = ditonton full, 0.3 = hanya 30% ditonton
df['watch_time_ratio'] = df['avg_watch_seconds'] / df['durasi_detik'].replace(0, np.nan)
df['watch_time_ratio'] = df['watch_time_ratio'].clip(0, 1)  # cap di 1.0

# Revenue per View (proxy kualitas penonton)
if 'Estimasi pendapatan (IDR)' in df.columns:
    df['revenue_per_view'] = df['Estimasi pendapatan (IDR)'] / views

# Subscriber Net (subscriber bersih per video)
sub_gain = 'Subscriber yang diperoleh'
sub_lost = 'Subscriber yang hilang'
if sub_gain in df.columns and sub_lost in df.columns:
    df['subscriber_net'] = df[sub_gain] - df[sub_lost].fillna(0)
    df['subscriber_net_per_view'] = df['subscriber_net'] / views

print(" Watch time & retention metrics dibuat:")
print("   watch_time_ratio, revenue_per_view, subscriber_net, subscriber_net_per_view")

 Watch time & retention metrics dibuat:
   watch_time_ratio, revenue_per_view, subscriber_net, subscriber_net_per_view


### 3.3 Impression & CTR Metrics

In [62]:
# CTR (sudah ada di data): Rasio klik-tayang dari tayangan (%)
df['ctr'] = df['Rasio klik-tayang dari tayangan (%)'] / 100  # ubah ke proporsi 0–1

# Impression-to-View Rate: seberapa efektif tayangan → jadi view
if 'Tayangan' in df.columns:
    df['impression_to_view_rate'] = df['Penayangan'] / df['Tayangan'].replace(0, np.nan)

print("Impression & CTR metrics dibuat:")
print("ctr (0-1), impression_to_view_rate")

Impression & CTR metrics dibuat:
ctr (0-1), impression_to_view_rate


### 3.4 Fitur Waktu Publikasi

In [63]:
if 'published_at' in df.columns:
    df['upload_hour']    = df['published_at'].dt.hour
    df['upload_day']     = df['published_at'].dt.dayofweek  # 0=Senin, 6=Minggu
    df['upload_day_name']= df['published_at'].dt.day_name()
    df['upload_month']   = df['published_at'].dt.month
    df['upload_year']    = df['published_at'].dt.year
    
    # Umur video dalam hari (dari tanggal upload sampai sekarang)
    today = pd.Timestamp.today().normalize()
    df['video_age_days'] = (today - df['published_at'].dt.normalize()).dt.days

    print(" Fitur waktu publikasi dibuat:")
    print("   upload_hour, upload_day (0=Senin), upload_day_name, upload_month, upload_year, video_age_days")

 Fitur waktu publikasi dibuat:
   upload_hour, upload_day (0=Senin), upload_day_name, upload_month, upload_year, video_age_days


### 3.5 Duration Bucket (Kategori Panjang Video)

In [64]:
def categorize_duration(seconds):
    """Kategorikan durasi video ke bucket yang bermakna untuk algoritma YouTube."""
    if pd.isna(seconds):
        return 'unknown'
    minutes = seconds / 60
    if minutes < 1:
        return 'shorts (<1 mnt)'
    elif minutes < 3:
        return 'pendek (1-3 mnt)'
    elif minutes < 7:
        return 'medium (3-7 mnt)'
    elif minutes < 15:
        return 'panjang (7-15 mnt)'
    else:
        return 'sangat panjang (>15 mnt)'

df['duration_bucket'] = df['durasi_detik'].apply(categorize_duration)

print("Duration bucket dibuat:")
print(df['duration_bucket'].value_counts())

Duration bucket dibuat:
duration_bucket
panjang (7-15 mnt)          2640
sangat panjang (>15 mnt)     190
pendek (1-3 mnt)             166
shorts (<1 mnt)                2
medium (3-7 mnt)               2
Name: count, dtype: int64


### 3.6 NLP Fitur dari Judul Video

In [65]:
# Kata-kata sensasional/clickbait yang umum di konten berita/geo-politik Indonesia
SENSATIONAL_WORDS = [
    'GEMPAR', 'HEBOH', 'NGAMUK', 'PANIK', 'MERINDING', 'GEGER', 'VIRAL',
    'MENGEJUTKAN', 'MENGEJUTKAN', 'BONGKAR', 'HANCUR', 'TERDIAM', 'KAGET',
    'MALU', 'TAKUT', 'MARAH', 'MENGGUNCANG', 'TERHINA', 'BOIKOT', 'PERANG',
    'GILA', 'LUAR BIASA', 'BERANI', 'NEKAT', 'HAJAR', 'BANTAI'
]

# Topik/cluster konten
TOPIC_KEYWORDS = {
    'israel_palestina': ['ISRAEL', 'PALESTINA', 'GAZA', 'HAMAS', 'IDF'],
    'malaysia':         ['MALAYSIA', 'MELAYU', 'JIRAN'],
    'militer_tni':      ['TNI', 'MILITER', 'ALUTSISTA', 'ANGKATAN', 'KAPAL PERANG', 'TANK', 'RUDAL'],
    'ekonomi_mineral':  ['MINERAL', 'NIKEL', 'BATU BARA', 'EKONOMI', 'INVESTASI', 'EKSPOR'],
    'diplomasi':        ['PRABOWO', 'DIPLOMATIK', 'PBB', 'SIDANG', 'HUBUNGAN', 'KTT'],
    'amerika_barat':    ['AMERIKA', 'AS', 'USA', 'BARAT', 'NATO', 'CIA'],
    'cina':             ['CINA', 'CHINA', 'TIONGKOK', 'XI JINPING'],
    'rusia':            ['RUSIA', 'RUSSIA', 'PUTIN'],
}

def extract_title_features(title):
    """Extract NLP features dari judul video."""
    if pd.isna(title):
        return {}
    
    title_upper = str(title).upper()
    title_str   = str(title)
    
    features = {}
    
    # 1. Panjang judul
    features['title_length']  = len(title_str)
    features['title_words']   = len(title_str.split())
    
    # 2. Jumlah kata sensasional
    features['sensational_count'] = sum(
        1 for w in SENSATIONAL_WORDS if w in title_upper
    )
    
    # 3. Apakah ada simbol/emoji clickbait
    features['has_symbol'] = int(bool(re.search(r'[‼⁉✅🚨🔥❗❓💥🇮🇩⚡]', title_str)))
    
    # 4. Proporsi huruf kapital (dari karakter huruf saja)
    letters = [c for c in title_str if c.isalpha()]
    if letters:
        features['caps_ratio'] = sum(1 for c in letters if c.isupper()) / len(letters)
    else:
        features['caps_ratio'] = 0.0
    
    # 5. Topic cluster (multi-label, pakai yang paling banyak keyword-nya)
    topic_scores = {}
    for topic, keywords in TOPIC_KEYWORDS.items():
        topic_scores[topic] = sum(1 for kw in keywords if kw in title_upper)
    
    best_topic = max(topic_scores, key=topic_scores.get)
    features['topic_cluster'] = best_topic if topic_scores[best_topic] > 0 else 'lainnya'
    features['topic_score']   = topic_scores[best_topic]
    
    # 6. Jumlah entitas negara/tokoh yang disebut
    all_entities = [kw for kws in TOPIC_KEYWORDS.values() for kw in kws]
    features['entity_count'] = sum(1 for e in all_entities if e in title_upper)
    
    return features

# Terapkan ke semua judul
title_features = df['Judul video'].apply(extract_title_features)
title_df = pd.DataFrame(title_features.tolist(), index=df.index)
df = pd.concat([df, title_df], axis=1)

print(" NLP fitur judul dibuat:")
print(" title_length, title_words, sensational_count, has_symbol,")
print(" caps_ratio, topic_cluster, topic_score, entity_count")
print(f"\nDistribusi topic_cluster:")
print(df['topic_cluster'].value_counts())

 NLP fitur judul dibuat:
 title_length, title_words, sensational_count, has_symbol,
 caps_ratio, topic_cluster, topic_score, entity_count

Distribusi topic_cluster:
topic_cluster
malaysia            1764
amerika_barat        388
israel_palestina     242
militer_tni          226
lainnya              174
diplomasi            108
rusia                 50
ekonomi_mineral       40
cina                   8
Name: count, dtype: int64


---
## 4. Performance Classification

Klasifikasi performa video berdasarkan **Views × Watch Time Ratio**.

Logika: Video bagus = banyak yang nonton DAN menontonnya lama.
- `1000 views × 2 mnt` > `500 views × 1 mnt`
- Score = `Penayangan × (avg_watch_seconds / 60)`

In [66]:

# PERFORMANCE SCORE: Views × Rata-rata waktu tonton (menit)

df['avg_watch_minutes'] = df['avg_watch_seconds'] / 60

# Score utama: Views × Avg Watch Minutes
# Contoh: 1000 views × 2 mnt = 2000 | 500 views × 1 mnt = 500
df['perf_score'] = df['Penayangan'] * df['avg_watch_minutes']

print("Distribusi perf_score:")
print(df['perf_score'].describe())

Distribusi perf_score:
count       3,000.00
mean      213,776.30
std       446,105.15
min         3,882.50
25%        33,248.10
50%        92,928.67
75%       207,149.40
max     6,773,554.47
Name: perf_score, dtype: float64


In [67]:
# PERFORMANCE CLASS berdasarkan persentil channel sendiri
# Lebih adil daripada threshold absolut karena
# channel besar dan kecil punya skala berbeda

p20 = df['perf_score'].quantile(0.20)  # 20% terbawah = Rendah
p50 = df['perf_score'].quantile(0.50)  # Median
p80 = df['perf_score'].quantile(0.80)  # 20% teratas = Bagus
p95 = df['perf_score'].quantile(0.95)  # 5% teratas = Viral

print(f"Persentil perf_score channel kamu:")
print(f"  P20 (batas Rendah):  {p20:,.0f}")
print(f"  P50 (Median):        {p50:,.0f}")
print(f"  P80 (batas Bagus):   {p80:,.0f}")
print(f"  P95 (batas Viral):   {p95:,.0f}")

def assign_performance_class(score):
    """
    Klasifikasi performa video berdasarkan Views × Watch Time.
    
    Tier:
       rendah   — P0–P20   (underperforming)
       sedang   — P20–P50  (di bawah median)
       bagus    — P50–P80  (di atas median)
       sangat_bagus — P80–P95  (top performers)
       viral    — P95+     (viral / outlier positif)
    """
    if pd.isna(score):
        return 'unknown'
    elif score < p20:
        return 'rendah'
    elif score < p50:
        return 'sedang'
    elif score < p80:
        return 'bagus'
    elif score < p95:
        return 'sangat_bagus'
    else:
        return 'viral'

df['performance_class'] = df['perf_score'].apply(assign_performance_class)

print("\n performance_class dibuat!")
print("\nDistribusi kelas:")
class_order = ['rendah', 'sedang', 'bagus', 'sangat_bagus', 'viral']
print(df['performance_class'].value_counts().reindex(class_order))

Persentil perf_score channel kamu:
  P20 (batas Rendah):  27,539
  P50 (Median):        92,929
  P80 (batas Bagus):   265,292
  P95 (batas Viral):   782,693

 performance_class dibuat!

Distribusi kelas:
performance_class
rendah          600
sedang          900
bagus           900
sangat_bagus    450
viral           150
Name: count, dtype: int64


In [68]:
# Visualisasi distribusi kelas performa
print("\n Ringkasan per kelas performa:")
print("-" * 80)

summary_cols = ['Penayangan', 'avg_watch_minutes', 'perf_score', 'ctr', 'like_rate', 'watch_time_ratio']
existing_summary = [c for c in summary_cols if c in df.columns]

summary = df.groupby('performance_class')[existing_summary].median().reindex(class_order)
print(summary.to_string())


 Ringkasan per kelas performa:
--------------------------------------------------------------------------------
                   Penayangan  avg_watch_minutes   perf_score  ctr  like_rate  watch_time_ratio
performance_class                                                                              
rendah               4,658.50               3.61    14,888.03 0.16       0.02              0.41
sedang              12,883.50               3.80    47,810.75 0.14       0.02              0.40
bagus               39,271.00               3.82   147,700.40 0.14       0.01              0.38
sangat_bagus       100,776.00               3.92   396,696.60 0.12       0.01              0.39
viral              296,402.00               4.13 1,242,448.90 0.12       0.01              0.36


### 4.1 Contoh Threshold Absolut (Opsional)

Kalau mau pakai threshold tetap (bukan persentil), uncomment kode di bawah ini dan sesuaikan angkanya:

In [69]:
# # Threshold absolut — sesuaikan dengan karakteristik channel kamu
# THRESHOLDS = {
#     'viral':       {'min_views': 500_000, 'min_watch_min': 3.0},
#     'sangat_bagus':{'min_views': 200_000, 'min_watch_min': 2.5},
#     'bagus':       {'min_views': 100_000, 'min_watch_min': 2.0},
#     'sedang':      {'min_views': 50_000,  'min_watch_min': 1.5},
#     'rendah':      {'min_views': 0,       'min_watch_min': 0.0},
# }

# def classify_absolute(row):
#     v = row.get('Penayangan', 0)
#     w = row.get('avg_watch_minutes', 0)
#     if v >= 500_000 and w >= 3.0: return 'viral'
#     elif v >= 200_000 and w >= 2.5: return 'sangat_bagus'
#     elif v >= 100_000 and w >= 2.0: return 'bagus'
#     elif v >= 50_000 and w >= 1.5: return 'sedang'
#     else: return 'rendah'

# df['performance_class_absolute'] = df.apply(classify_absolute, axis=1)
print("(Threshold absolut — uncomment untuk dipakai)")

(Threshold absolut — uncomment untuk dipakai)


---
## 5. Ringkasan Fitur Final

In [70]:
# Fitur yang akan dipakai untuk ML
ML_FEATURES = [
    # Identitas
    'Konten',           # video ID
    'Judul video',
    'published_at',
    
    # Raw Metrics (sudah ada di data)
    'Penayangan',
    'Tayangan',
    'durasi_detik',
    'avg_watch_seconds',
    'Poin hype',
    'RPM (IDR)',
    'CPM (IDR)',
    'Persentase penayangan rata-rata (%)',
    
    # Fitur Engineered
    'like_rate',
    'comment_rate',
    'share_rate',
    'dislike_ratio',
    'engagement_rate',
    'watch_time_ratio',
    'revenue_per_view',
    'subscriber_net',
    'subscriber_net_per_view',
    'ctr',
    'impression_to_view_rate',
    'upload_hour',
    'upload_day',
    'upload_day_name',
    'upload_month',
    'upload_year',
    'video_age_days',
    'duration_bucket',
    
    # NLP Features
    'title_length',
    'title_words',
    'sensational_count',
    'has_symbol',
    'caps_ratio',
    'topic_cluster',
    'topic_score',
    'entity_count',
    
    # Target Variable
    'perf_score',
    'performance_class',
]

existing_ml_features = [f for f in ML_FEATURES if f in df.columns]
df_ml = df[existing_ml_features].copy()

print(f" Dataset ML siap: {df_ml.shape[0]} video × {df_ml.shape[1]} fitur")
print("\nFitur tersedia:")
for f in existing_ml_features:
    null_pct = df_ml[f].isnull().mean() * 100
    dtype = df_ml[f].dtype
    print(f"  {f:<40} {str(dtype):<12} null: {null_pct:.1f}%")

 Dataset ML siap: 3000 video × 39 fitur

Fitur tersedia:
  Konten                                   str          null: 0.0%
  Judul video                              str          null: 0.0%
  published_at                             datetime64[us] null: 0.0%
  Penayangan                               float64      null: 0.0%
  Tayangan                                 float64      null: 0.0%
  durasi_detik                             float64      null: 0.0%
  avg_watch_seconds                        float64      null: 0.0%
  Poin hype                                float64      null: 0.0%
  RPM (IDR)                                float64      null: 0.7%
  CPM (IDR)                                float64      null: 5.9%
  Persentase penayangan rata-rata (%)      float64      null: 0.0%
  like_rate                                float64      null: 0.0%
  comment_rate                             float64      null: 0.0%
  share_rate                               float64      null: 0.0%
  d

---
## 6. Simpan Hasil

In [71]:
# Simpan dataset bersih + fitur lengkap
OUTPUT_FULL = 'data_cleaned_full.csv'
OUTPUT_ML   = 'data_ml_features.csv'

df.to_csv(OUTPUT_FULL, index=False)
df_ml.to_csv(OUTPUT_ML, index=False)

print(f" Disimpan:")
print(f"   {OUTPUT_FULL}  → {df.shape[0]} baris, {df.shape[1]} kolom (semua kolom)")
print(f"   {OUTPUT_ML}    → {df_ml.shape[0]} baris, {df_ml.shape[1]} kolom (fitur ML saja)")

 Disimpan:
   data_cleaned_full.csv  → 3000 baris, 90 kolom (semua kolom)
   data_ml_features.csv    → 3000 baris, 39 kolom (fitur ML saja)


In [77]:
# Preview 5 baris terakhir
print("\n🔍 Sample data ML:")
preview_cols = ['Judul video', 'Penayangan', 'avg_watch_minutes', 
                'perf_score', 'performance_class', 'topic_cluster', 'ctr']
existing_preview = [c for c in preview_cols if c in df_ml.columns]
df_ml[existing_preview].head(10)


🔍 Sample data ML:


,Judul video,Penayangan,perf_score,performance_class,topic_cluster,ctr
0,NEGARA ARAB & MALAYSIA TERDIAM MELIHAT INDONES...,"915,959.00","4,503,465.08",viral,israel_palestina,0.12
1,INDONESIA BIKIN ISRAEL NGAMUK‼ SAAT ARAB TAKUT...,"319,187.00","1,271,428.22",viral,israel_palestina,0.14
2,"MALAYSIA KENA ""CERAMAH"" WARGANYA SENDIRI: JANG...","286,864.00","1,415,195.73",viral,malaysia,0.16
3,GILA KEBERANIAN INDONESIA BIKIN NETANYAHU PANI...,"280,836.00","1,245,039.60",viral,israel_palestina,0.11
4,"NETANYAHU NGAMUK‼ ANCAM PERANGI INDONESIA, MAL...","272,539.00","1,162,833.07",viral,malaysia,0.12
5,"HEBOH‼ MALAYSIA TANTANG PRABOWO, TAPI BALASANN...","234,153.00","1,018,565.55",viral,malaysia,0.12
6,LAPINDO JADI LAUTAN EMAS‼ AMERIKA & EROPA NGIL...,"222,453.00","689,604.30",sangat_bagus,amerika_barat,0.14
7,GEMPAR‼ MALAYSIA PANIK‼ PRABOWO TEGAS MINTA IN...,"204,148.00","1,061,569.60",viral,malaysia,0.11
8,GEMPAR‼ PRABOWO BAWA PULANG PUSAKA BANGSA SETE...,"191,662.00","833,729.70",viral,amerika_barat,0.09
9,BANGGA‼ RIBUAN BUS CANGGIH RI SERBU BANGLADESH...,"180,861.00","693,300.50",sangat_bagus,malaysia,0.12


---
## 7. Catatan & Next Steps

### Yang sudah selesai:
-  Hapus baris Total & baris kosong
-  Drop kolom >80% null
-  Konversi tipe data (waktu tonton, durasi, datetime)
-  Engineering: engagement rates, watch time ratio, CTR, subscriber net
-  Fitur waktu publikasi (jam, hari, bulan, umur video)
-  Duration bucket
-  NLP fitur dari judul (topic cluster, sensational count, dll)
-  **Performance class** berbasis Views × Watch Time (5 kelas)

### Next Steps untuk ML:
1. **Model 1 — Classifier**: XGBoost/LightGBM untuk prediksi `performance_class`
2. **Model 2 — Regressor**: Prediksi total views
3. **Model 3 — SHAP Analysis**: Kenapa video underperform?
4. **Data tambahan**: Export time series harian per video dari YouTube Studio untuk rolling average & velocity decay


In [73]:
df.duplicated()  # Mengembalikan boolean untuk setiap baris
df[df.duplicated()].shape[0]  # Hitung jumlah duplikat

1641

In [74]:
# Semua baris yang duplikat (tidak termasuk kemunculan pertama)
df[df.duplicated()]

# Semua baris yang duplikat (termasuk kemunculan pertama)
df[df.duplicated(keep=False)]

,Konten,Judul video,Waktu publikasi video,Durasi,Penayangan tak dilewati,Rata-rata durasi tonton,Persentase penayangan rata-rata (%),Diviralkan,Poin hype,Subscriber yang diperoleh,Subscriber yang hilang,Suka,Tidak suka,Suka (vs. tidak suka) (%),Pembagian,Komentar ditambahkan,Pendapatan transaksi (IDR),Transaksi,YouTube Premium (IDR),Iklan Halaman Tonton (IDR),Estimasi pendapatan DoubleClick (IDR),Estimasi pendapatan AdSense (IDR),Pendapatan iklan YouTube (IDR),Tayangan iklan,CPM berbasis-pemutaran (IDR),...,engagement_rate,watch_time_ratio,revenue_per_view,subscriber_net,subscriber_net_per_view,ctr,impression_to_view_rate,upload_hour,upload_day,upload_day_name,upload_month,upload_year,video_age_days,duration_bucket,title_length,title_words,sensational_count,has_symbol,caps_ratio,topic_cluster,topic_score,entity_count,avg_watch_minutes,perf_score,performance_class
0,KAqnuJ0W8Jo,NEGARA ARAB & MALAYSIA TERDIAM MELIHAT INDONES...,"Aug 23, 2025",740.00,"915,959.00",0:04:55,39.94,0.00,0.00,"5,516.00",138.00,"13,544.00",184.00,98.66,"2,284.00","1,840.00",0.00,0.00,"137,839.84","4,058,011.66",0.00,"4,058,011.66","7,378,203.32","832,574.00","11,672.97",...,0.02,0.40,4.58,"5,378.00",0.01,0.12,0.16,0,5,Saturday,8,2025,237,panjang (7-15 mnt),96,14,1,0,1.00,israel_palestina,2,3,4.92,"4,503,465.08",viral
1,a5VBaiz4yS0,INDONESIA BIKIN ISRAEL NGAMUK‼ SAAT ARAB TAKUT...,"Aug 24, 2025",649.00,"319,187.00",0:03:59,36.89,0.00,0.00,820.00,30.00,"3,033.00",43.00,98.60,287.00,435.00,0.00,0.00,"45,200.91","1,006,901.20",0.00,"1,006,901.20","1,830,729.43","209,284.00","11,520.83",...,0.01,0.37,3.30,790.00,0.00,0.14,0.21,0,6,Sunday,8,2025,236,panjang (7-15 mnt),93,14,3,1,1.00,israel_palestina,2,4,3.98,"1,271,428.22",viral
2,PNIbEl5676s,"MALAYSIA KENA ""CERAMAH"" WARGANYA SENDIRI: JANG...","Dec 23, 2025",712.00,"286,864.00",0:04:56,41.59,0.00,0.00,511.00,63.00,"3,433.00",48.00,98.62,137.00,"1,498.00",0.00,0.00,"21,277.47","984,382.82",0.00,"984,382.82","1,789,786.94","215,690.00","10,451.68",...,0.02,0.42,3.51,448.00,0.00,0.16,0.19,0,1,Tuesday,12,2025,115,panjang (7-15 mnt),64,8,0,0,1.00,malaysia,1,1,4.93,"1,415,195.73",viral
3,jj4tR1MXgGM,GILA KEBERANIAN INDONESIA BIKIN NETANYAHU PANI...,"Sep 26, 2025",752.00,"280,836.00",0:04:26,35.43,0.00,0.00,950.00,57.00,"3,076.00",51.00,98.37,214.00,505.00,0.00,5.00,"5,172.97","858,249.02",0.00,"858,249.02","1,560,452.98","255,225.00","7,205.87",...,0.01,0.35,3.10,893.00,0.00,0.11,0.13,0,4,Friday,9,2025,203,panjang (7-15 mnt),87,12,3,0,1.00,israel_palestina,1,2,4.43,"1,245,039.60",viral
4,Sj1GSN-q5N8,"NETANYAHU NGAMUK‼ ANCAM PERANGI INDONESIA, MAL...","Aug 26, 2025",900.00,"272,539.00",0:04:16,28.49,0.00,0.00,"1,213.00",32.00,"2,448.00",222.00,91.69,"1,046.00","2,410.00",0.00,2.00,"16,682.47","1,177,888.70",0.00,"1,177,888.70","2,141,615.56","266,076.00","9,527.23",...,0.02,0.28,4.40,"1,181.00",0.00,0.12,0.14,0,1,Tuesday,8,2025,234,sangat panjang (>15 mnt),88,12,3,1,1.00,malaysia,1,1,4.27,"1,162,833.07",viral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3000,rAaTvV8NScE,MALAYSIA SEBUT INDONESIA MENUJU KEHANCURAN‼ KI...,"Sep 6, 2025",660.00,"26,415.00",0:04:13,38.46,0.00,0.00,31.00,13.00,428.00,13.00,97.05,23.00,164.00,0.00,3.00,"2,333.73","103,002.49",0.00,"103,002.49","187,277.14","24,558.00","9,097.75",...,0.02,0.38,4.08,18.00,0.00,0.14,0.17,0,5,Saturday,9,2025,223,panjang (7-15 mnt),98,14,1,1,1.00,malaysia,1,2,4.22,"111,383.25",bagus
3001,wHJd6SU_MHA,"3 WILAYAH INI INGIN MERDEKA DARI INDONESIA, NO...","Dec 5, 2025",730.00,"26,367.00",0:04:19,35.58,0.00,0.00,37.00,3.00,183.00,13.00,93.37,48.00,88.00,0.00,0.00,457.78,"117,528.64",0.00,"117,528.64","213,688.35","29,021.00","8,344.27",...,0.01,0.35,4.47,34.00,0.00,0.14,0.16,0,4,Friday,12,2025,133,panjang (7-15 mnt),60,11,1,0,1.00,lainnya,0,0,4.32,"113,817.55",bagus
3002,cciwl1weyfI,GEGER‼ AUSTRALIA AKUI I

In [80]:
# Tampilkan metrik dengan judul yang duplikat
print(f"Total judul yang duplikat: {df.duplicated(subset=['Judul video']).sum()}")
print(f"Total baris dengan judul duplikat: {df.duplicated(subset=['Judul video'], keep=False).sum()}\n")

# Tampilkan data
df[df.duplicated(subset=['Judul video'], keep=False)].sort_values('Judul video')

Total judul yang duplikat: 1882
Total baris dengan judul duplikat: 3000



,Konten,Judul video,Waktu publikasi video,Durasi,Penayangan tak dilewati,Rata-rata durasi tonton,Persentase penayangan rata-rata (%),Diviralkan,Poin hype,Subscriber yang diperoleh,Subscriber yang hilang,Suka,Tidak suka,Suka (vs. tidak suka) (%),Pembagian,Komentar ditambahkan,Pendapatan transaksi (IDR),Transaksi,YouTube Premium (IDR),Iklan Halaman Tonton (IDR),Estimasi pendapatan DoubleClick (IDR),Estimasi pendapatan AdSense (IDR),Pendapatan iklan YouTube (IDR),Tayangan iklan,CPM berbasis-pemutaran (IDR),...,engagement_rate,watch_time_ratio,revenue_per_view,subscriber_net,subscriber_net_per_view,ctr,impression_to_view_rate,upload_hour,upload_day,upload_day_name,upload_month,upload_year,video_age_days,duration_bucket,title_length,title_words,sensational_count,has_symbol,caps_ratio,topic_cluster,topic_score,entity_count,avg_watch_minutes,perf_score,performance_class
788,GKF6T1ZECEE,2 NEGARA PALING ANGKUH ASEAN RESMI RUNTUH TOTA...,"Mar 15, 2026",600.00,"5,341.00",0:04:06,41.11,0.00,0.00,2.00,2.00,105.00,3.00,97.22,14.00,34.00,0.00,0.00,389.19,"32,948.08",0.00,"32,948.08","59,821.86","6,648.00","11,314.90",...,0.03,0.41,6.24,0.00,0.00,0.17,0.20,0,6,Sunday,3,2026,33,panjang (7-15 mnt),99,13,1,0,1.00,malaysia,1,2,4.10,"21,898.10",rendah
1790,GKF6T1ZECEE,2 NEGARA PALING ANGKUH ASEAN RESMI RUNTUH TOTA...,"Mar 15, 2026",600.00,"5,341.00",0:04:06,41.11,0.00,0.00,2.00,2.00,105.00,3.00,97.22,14.00,34.00,0.00,0.00,389.19,"32,948.08",0.00,"32,948.08","59,821.86","6,648.00","11,314.90",...,0.03,0.41,6.24,0.00,0.00,0.17,0.20,0,6,Sunday,3,2026,33,panjang (7-15 mnt),99,13,1,0,1.00,malaysia,1,2,4.10,"21,898.10",rendah
981,lduTiErgFEY,20 NEGARA MERINDING! TNI AL KIRIM FREGAT SILUM...,"Mar 5, 2026",540.00,"2,727.00",0:03:24,37.78,0.00,0.00,1.00,2.00,39.00,1.00,97.50,0.00,20.00,0.00,0.00,132.21,"13,789.57",0.00,"13,789.57","25,100.61","2,913.00","9,827.96",...,0.02,0.38,5.11,-1.00,-0.00,0.18,0.25,0,3,Thursday,3,2026,43,panjang (7-15 mnt),87,14,1,0,1.00,malaysia,1,2,3.40,"9,271.80",rendah
1983,lduTiErgFEY,20 NEGARA MERINDING! TNI AL KIRIM FREGAT SILUM...,"Mar 5, 2026",540.00,"2,727.00",0:03:24,37.78,0.00,0.00,1.00,2.00,39.00,1.00,97.50,0.00,20.00,0.00,0.00,132.21,"13,789.57",0.00,"13,789.57","25,100.61","2,913.00","9,827.96",...,0.02,0.38,5.11,-1.00,-0.00,0.18,0.25,0,3,Thursday,3,2026,43,panjang (7-15 mnt),87,14,1,0,1.00,malaysia,1,2,3.40,"9,271.80",rendah
1729,IqPnC9CWTqA,22 NEGARA SERENTAK DEKLARASIKAN PERANG‼️ MALAY...,"Mar 7, 2026",540.00,"7,248.00",0:03:45,41.80,0.00,0.00,1.00,0.00,81.00,2.00,97.59,17.00,24.00,0.00,0.00,174.53,"38,002.41",0.00,"38,002.41","69,143.20","8,172.00","10,003.36",...,0.02,0.42,5.27,1.00,0.00,0.16,0.19,0,5,Saturday,3,2026,41,panjang (7-15 mnt),86,11,1,1,1.00,malaysia,1,2,3.75,"27,180.00",rendah
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
840,72Nf9WUc1Kk,"[STRATIFIKASI SOSIAL #1] : Definisi, Kriteria,...","Feb 6, 2021",499.00,"4,363.00",0:02:23,28.85,0.00,0.00,10.00,2.00,58.00,2.00,96.67,212.00,4.00,0.00,0.00,66.72,"47,868.11",0.00,"47,868.11","87,072.76","7,203.00","15,842.93",...,0.06,0.29,10.99,8.00,0.00,0.19,0.39,0,5,Saturday,2,2021,1896,panjang (7-15 mnt),93,13,0,0,0.46,amerika_barat,1,1,2.38,"10,398.48",rendah
1951,uGfQ1TV8WIA,"​MEMANAS‼️ MALAYSIA PANIK, FILIPINA REBUT SABA...","Apr 1, 2026",600.00,"3,055.00",0:03:42,37.03,0.00,0.00,1.00,0.00,47.00,1.00,97.92,4.00,30.00,0.00,0.00,183.34,"14,213.11",0.00,"14,213.11","25,858.99","3,224.00","9,114.91",...,0.03,0.37,4.71,1.00,0.00,0.17,0.20,0,2,Wednesday,4,2026,16,panjang (7-15 mnt),79,10,1,1,1.00,malaysia,1,2,3.70,"11,303.50",rendah
949,uGfQ1TV8WIA,"​MEMANAS‼️ MALAYSIA PANIK, FILIPINA REBUT SABA...","Apr 1, 2026",600.00,"3,055.00",0:03:42,37.03,0.00,0.00,1.00,0.00,47.00,1.00,97.92,4.00,30.00,0.00,0.00,183.34,"14,213.11",0.00,"14,213.11","25,858.99","3,224.00","9,114.91",...,0.03,0.37,4.71,1.00,0.00,0.17,0.